# Hybrid Search: BM25 + Cosine (Giga-Embeddings + ChromaDB)

**Pipeline:**
1. Определяем документы и запрос
2. Лемматизируем документы → векторизируем → кладём в ChromaDB
3. Лемматизируем запрос → векторизируем
4. Гибридный поиск: BM25 + косинусное сходство

In [11]:
!pip install flash_attn==2.8.1 rank_bm25 pymorphy3 chromadb transformers==4.51.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 119.4 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [1]:
import torch
import numpy as np
import chromadb
import pymorphy3
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModel

In [2]:
# --- Лемматизатор (pymorphy3) ---
# pip install pymorphy3 pymorphy3-dicts-ru
morph = pymorphy3.MorphAnalyzer()

def lemmatize(text: str) -> str:
    words = text.lower().split()
    return ' '.join(morph.parse(word)[0].normal_form for word in words)

# --- Инструкция для query-эмбеддинга ---
def get_detailed_instruct(task_description: str, query: str) -> str:
    return f'Instruct: {task_description}\nQuery: {query}'

In [37]:
# --- Корпус документов ---
documents = [
    "утвержден план по покупке 10000 евро",
    "реализован ai agent по покупке рубля по выгодной цене",
    "выдать кредиты на 10 миллионов рублей",
    "построить новый офис с бюджетом 15 миллионов рублей до 12.01.27",
    "повысить запасы евро",
]

# --- Запрос ---
# task = 'Given a web search query, retrieve relevant passages that answer the query'
task = ""
query = 'повысить запасы рубля'

print(f"Documents : {len(documents)}")
print(f"Query     : {query}")

Documents : 5
Query     : повысить запасы рубля


In [4]:
# --- Патч для transformers 5.x: добавляем 'default' в ROPE_INIT_FUNCTIONS ---
# В transformers>=5.0 ключ 'default' убрали из словаря, но модель его ожидает
from transformers.modeling_rope_utils import ROPE_INIT_FUNCTIONS

def _compute_default_rope_parameters(config, device=None, seq_len=None, **rope_kwargs):
    base = config.rope_theta
    partial_rotary_factor = getattr(config, 'partial_rotary_factor', 1.0)
    head_dim = getattr(config, 'head_dim', config.hidden_size // config.num_attention_heads)
    dim = int(head_dim * partial_rotary_factor)
    inv_freq = 1.0 / (
        base ** (torch.arange(0, dim, 2, dtype=torch.int64).float().to(device) / dim)
    )
    return inv_freq, 1.0  # inv_freq, attention_scaling

if 'default' not in ROPE_INIT_FUNCTIONS:
    ROPE_INIT_FUNCTIONS['default'] = _compute_default_rope_parameters
    print("Patched: 'default' RoPE → ROPE_INIT_FUNCTIONS")
else:
    print("No patch needed (transformers version already supports 'default' RoPE)")

No patch needed (transformers version already supports 'default' RoPE)


In [17]:
tokenizer = AutoTokenizer.from_pretrained(
    'ai-sage/Giga-Embeddings-instruct',
    trust_remote_code=True
)
model = AutoModel.from_pretrained(
    'ai-sage/Giga-Embeddings-instruct',
    attn_implementation="eager",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)
model.eval()
model.cuda()
print(f"Model device: {next(model.parameters()).device}")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Model device: cuda:0


In [38]:
# --- Лемматизация документов ---
lemmatized_docs = [lemmatize(doc) for doc in documents]

print("Lemmatized documents:")
for i, (orig, lemm) in enumerate(zip(documents, lemmatized_docs)):
    print(f"  [{i}] {orig}")
    print(f"      → {lemm}")

Lemmatized documents:
  [0] утвержден план по покупке 10000 евро
      → утвердить план по покупка 10000 евро
  [1] реализован ai agent по покупке рубля по выгодной цене
      → реализовать ai agent по покупка рубль по выгодный цена
  [2] выдать кредиты на 10 миллионов рублей
      → выдать кредит на 10 миллион рубль
  [3] построить новый офис с бюджетом 15 миллионов рублей до 12.01.27
      → построить новый офис с бюджет 15 миллион рубль до 12.01.27
  [4] повысить запасы евро
      → повысить запас евро


In [25]:
# --- Лемматизация запроса ---
lemmatized_query = lemmatize(query)
instructed_query = get_detailed_instruct(task, lemmatized_query)

print(f"Original query    : {query}")
print(f"Lemmatized query  : {lemmatized_query}")
print(f"Instructed query  : {instructed_query}")

Original query    : повысить запасы рубля
Lemmatized query  : повысить запас рубль
Instructed query  : Instruct: 
Query: повысить запас рубль


In [39]:
# --- Функция векторизации ---
MAX_LENGTH = 4096

def embed(text: str) -> np.ndarray:
    encoded = tokenizer(
        text,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors='pt',
    ).to(model.device)
    with torch.no_grad():
        emb = model(**encoded, return_embeddings=True)  # (1, hidden_dim)
    return emb.squeeze(0).float().cpu().numpy()

# --- Векторизация документов ---
doc_embeddings = []
for i, lemm_doc in enumerate(lemmatized_docs):
    vec = embed(lemm_doc)
    doc_embeddings.append(vec)
    print(f"[{i}] embedded, shape={vec.shape}")

doc_embeddings = np.array(doc_embeddings)  # (N, hidden_dim)
print(f"\nDoc matrix shape: {doc_embeddings.shape}")

[0] embedded, shape=(2048,)
[1] embedded, shape=(2048,)
[2] embedded, shape=(2048,)
[3] embedded, shape=(2048,)
[4] embedded, shape=(2048,)

Doc matrix shape: (5, 2048)


In [40]:
# --- ChromaDB: создание коллекции и загрузка документов ---
client = chromadb.EphemeralClient()
client.delete_collection("documents")

collection = client.create_collection(
    name="documents",
    metadata={"hnsw:space": "cosine"}
)


collection.add(
    ids=[str(i) for i in range(len(documents))],
    embeddings=doc_embeddings.tolist(),
    documents=documents,
    metadatas=[{'lemmatized': lemm} for lemm in lemmatized_docs]
)

print(f"ChromaDB collection '{collection.name}': {collection.count()} documents")

ChromaDB collection 'documents': 5 documents


In [33]:
# --- Векторизация запроса ---
query_embedding = embed(instructed_query)  # (hidden_dim,)
print(f"Query embedding shape: {query_embedding.shape}")

Query embedding shape: (2048,)


In [41]:
# --- BM25 ---
tokenized_corpus = [doc.split() for doc in lemmatized_docs]
tokenized_query_tokens = lemmatized_query.split()

bm25 = BM25Okapi(tokenized_corpus)
bm25_raw = bm25.get_scores(tokenized_query_tokens)

# Нормализация в [0, 1]
bm25_norm = (bm25_raw - bm25_raw.min()) / (bm25_raw.max() - bm25_raw.min() + 1e-9)

print("BM25 raw scores      :", np.round(bm25_raw, 4))
print("BM25 normalized [0,1]:", np.round(bm25_norm, 4))

BM25 raw scores      : [0.     0.2035 0.2462 0.1924 2.9354]
BM25 normalized [0,1]: [0.     0.0693 0.0839 0.0655 1.    ]


In [42]:
# --- Cosine similarity (через ChromaDB) ---
chroma_results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=len(documents),
    include=['distances', 'documents', 'metadatas']
)

# ChromaDB с cosine space возвращает distance = 1 - cosine_similarity
chroma_ids = [int(i) for i in chroma_results['ids'][0]]
chroma_distances = chroma_results['distances'][0]

# Восстанавливаем порядок: косинусное сходство по оригинальному индексу
cosine_scores = np.zeros(len(documents))
for idx, dist in zip(chroma_ids, chroma_distances):
    cosine_scores[idx] = 1.0 - dist  # cosine_sim = 1 - cosine_distance

cosine_norm = (cosine_scores - cosine_scores.min()) / (cosine_scores.max() - cosine_scores.min() + 1e-9)

print("Cosine scores      :", np.round(cosine_scores, 4))
print("Cosine norm [0,1]  :", np.round(cosine_norm, 4))

Cosine scores      : [0.4996 0.607  0.4929 0.5069 0.7248]
Cosine norm [0,1]  : [0.029  0.4922 0.     0.0606 1.    ]


In [45]:
# --- Гибридный поиск: alpha * cosine + (1-alpha) * BM25 ---
alpha = 0.7  # вес косинуса; 1-alpha — вес BM25

hybrid_scores = alpha * cosine_norm + (1 - alpha) * bm25_norm
ranked = np.argsort(hybrid_scores)[::-1]

print(f"Query: \"{query}\"")
print(f"alpha={alpha} (cosine) | {1-alpha} (BM25)\n")
print(f"{'Rank':<6} {'Hybrid':>8} {'Cosine':>8} {'BM25':>8}   Document")
print('-' * 80)
for rank, idx in enumerate(ranked):
    print(f"{rank+1:<6} {hybrid_scores[idx]:>8.4f} {cosine_norm[idx]:>8.4f} "
          f"{bm25_norm[idx]:>8.4f}   {documents[idx]}")

Query: "повысить запасы рубля"
alpha=0.7 (cosine) | 0.30000000000000004 (BM25)

Rank     Hybrid   Cosine     BM25   Document
--------------------------------------------------------------------------------
1        1.0000   1.0000   1.0000   повысить запасы евро
2        0.3653   0.4922   0.0693   реализован ai agent по покупке рубля по выгодной цене
3        0.0621   0.0606   0.0655   построить новый офис с бюджетом 15 миллионов рублей до 12.01.27
4        0.0252   0.0000   0.0839   выдать кредиты на 10 миллионов рублей
5        0.0203   0.0290   0.0000   утвержден план по покупке 10000 евро
